# Topic 4 practical: What makes a car fuel-efficient?
### A typing lab with Python and scikit-learn
**ICDS · Strathmore University · @iLabAfrica**

**Your job:** predict a car's city fuel efficiency in miles per gallon (MPG). Higher MPG means the car travels farther per gallon.

By the end, you will have fitted simple, multiple and polynomial regression, compared them with a decision tree and a random forest, tuned one model, and explained its errors.

**Plan:** allow about 2–2½ hours with discussion. If time is short, complete Parts A–D in class and finish Part E afterwards. Part F is an optional extension.

| Part | Activity | Suggested time |
|---|---|---|
| A | Explore and prepare | 20 min |
| B | Fit models and inspect their shapes | 35–40 min |
| C | Compare with cross-validation | 15–20 min |
| D | Tune and select | 15 min |
| E | Test, visualise and interpret | 25–30 min |
| F | Optional challenges | After class |

### Open in Colab
At [colab.research.google.com](https://colab.research.google.com/), choose **File → Upload notebook** and upload this `.ipynb`. Use a standard CPU runtime. No GPU, package installation or separate dataset download is needed in a normal Colab runtime.

### How to work
- **Read, predict, type, run, explain.** Use `Shift + Enter` to run a cell.
- Type your own code in cells marked **TYPE**. These contain prompts, not completed answers.
- Discuss the **Predict first** questions before executing a model.
- Use the hints only after an attempt. Ask your partner to explain an error before taking over their keyboard.
- Checkpoint cells check selected properties; a pass does not prove every modelling choice is correct.
- Run cells from top to bottom. **Run all** at the start will stop at the first unfinished checkpoint. This is intentional.

**Name:** …  **Partner:** …  **Date:** …

## Setup — run once, no typing required
The following cell unpacks a copy of the source data into `auto_mpg.csv` and imports the tools. It contains no fitted models or completed exercise answers. The long data string is just an embedded dataset.

The dataset is historical teaching data. We will assess predictions for randomly held-out records from the same collection. This exercise does not validate predictions for today's vehicles or future model years.

In [ ]:
#@title Setup: unpack data and import tools (run once)
# Supplied setup: run this cell once.
import base64
import io
import zlib
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.base import clone
from sklearn.model_selection import train_test_split, KFold, cross_validate, GridSearchCV
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, PolynomialFeatures, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

_DATA = "eJyNXMuW27qOnfsrNOt7Vlfqii9JnPWf3CXbiksnsuWW5Hrk65sEQBKkXOkzSSUrgkjisQFsQnW9X15OX9N4Ow/L+nIe1/vUn4brcNte3uZlHe7zx7C8fAzj5W176U+nYRqWfhvn28t1Pg/Tf76GfnmZl/Ey3l5O/fKfW38dDqJ7rV+6F1W37qdQtftTmVr7f0j3R+t+vpzehvdlnoat8n8bpmmorv00Hh8HYVDaeDnR+H+pxir/D/FqUPr4GE+/qvXX19Qvvyol67gm/BQgq7RqUIrWvE9f1/mxvVVrv7kVx81ttaGtai6msq3211O1DMdhqtZ1O4iWJPz/C40S2vp/1GF7P+flXG1OLbc5HEdLeMT67WmlBT5fs+cv/dR/jkNlancajULG/5SyBiH4h90rcLzenWiUgS1JWFYrOEUX9hUV8POxfFXjOKaFDCwEQhp+pN3d59s29qfq1G/OQrc+WsiCheBP1YHu4kpeZ/312K9rf56X6nyfolQH2m3RKxqVLXWez5fBeUTv/OF2GZZqHcIWlUZ/ALmmtrBaadj/Oj3OfeUejWqv62TYthGgQVNq8OpO6Jbtl2l+phFVd022TXS/Yd2cH1Uf/WW+Vf9aP/46SC+rncf5U1l4gWrBTwyKqpdt/pq3vjrNy3zrncs793VmkP6hhtwDBTv0QrOz3fmxbsOCDu9FvCasd0rZtjoT8UZwUXwbtoMU8LQEdXSwgOkgPJvMCa/9+7C4wx1kC0eBF3cdeBQEslOPwZOc+2193Kr7ZER9kE16XIOyOgWrUEzIl/d5+rU6XQ23yulHVA49Hp+DM/C5vx3Q8ZziYHtwmAYV1wb5+/C4DPPmwkNHPQPAgP9Jne1OvvSP81iJuq6mNb4dgjyYxWRvX/v+WFk7hIMIUBgaUkpF2EUPH68fldOkDFolG8A+Gt0xe6MNLstwdYFzwNN5700R2sAPcLpkhZ/S1PFpOCQaTjsXzt7uXfirOnn8Ewz/pMCnO1ChCv6A0XV27zpYDnsArrpV8HQM4TenP/fo+f9xBbFzhQ41CPGKSnFaiY4pssh7Hy59JZXbUXSBFDsSXoW6EWXsBAEIGacf9DqLz5Ie7+NtcxFt0ffxfDWaCU2qwuOZmRp8nmAQEEArm51gn0qqk4vL+Yr5wUlj+oLVlJL2m/Pvkp9l0qBsRZkmCrPkgrmi2x1Pya7LZPzxrg6+HRZHPGX5VUvAU0x54k/JBeG0xTyBVhVRJ0WeqI7L/Li89de0JASVAXcTRkfnFH/IgTyj69o2mdmeJDT5yrJMB1IWsDxtFOPAgX5/mhG44Y3peCDVokdFncAGT/Pjtrml1v99jMvwVNivZdD7k3DQzdr/7JeR5MhwBg4IISttIzOtJAiv1vu8bP3RHZeSjWRxBmApdd1lQVDEGa7KPSy6S0pSfLc/3RmP43JOW01uKTAidGa/q4uB/nY5SEUoCtZoMJhrFszi5TosJ2+0U393CnFAk5ADAhCRQ8CbgpR8me+uDHMlR30ACNIvLRZEAJGY/izuKGUMh3Lhadg6YKisG8P2L19+jv3mAE8fD0rgmwHcMU20iuk1A6Jp6gEmDwrhCAzRgGoagT4YpAgl4WmOqZAOXL7UbI0sX0Kp7bKmiGlWhEML9G1Ms1lEnHwa95l/X5PINqGD3NUkb/1y3uZ7xFefIzsyBkREC/6WqrV52pKQSuei4hV/KFwsO9f2dR8qdcDyNksY0ZENrpQ7Mubd0r1gb02QSPhfLY8bhE0I1Bz52tSXyH+KfAoqqCT0bYXMsUuAhwgVdvhtMZ7DJKYOFZbaw2ToR1gHQ7WTCEsV1Ti0MVgvYFsiQd26UdRl0VohRF2V6uBuLfQHjqdNLbNTYWk8DWt/XFJOlJniQR3aNJngPJ3X63wcXS503u7cseuqZf7qp6EAWIviYHusydBuy9fqmtPqNnx4oERZRDuFsYJFssLaRQXnv/a/XcuwfEqP7fchmo7rs7P5YiydZimAd7wuV6XqRT5P+qf55hZd1uIlqbN05tGvoUhP5l/6W6gAUPJZqmzbbPlvKxaWjFLdC+nEKhUhkkL3fa6ENkOegUAGUdWA9yDmFdGuhSCxEL2p2LetZfGeFfskQ8DXBHSVorPZSstw6x+Th/BicwkjFMBfoUzECBTpEniBxSXVUYR4EcJdmUsSih0F/cvUDVMagutj1/HlCwJ3gRAbKuU9xKLEvhbHqGiegDnkpqauc/cyDMhIGNBFxeg9DbfNx70yqQr7Dl5UWV1mq2goni1qMa7CGm+qesMiiCVwxlrLqEf1xPOfYaxq0ellEArqc6r3cZehUcbIWJMfqYA+VspmWIShRoWw3J8PipuT28HksHc8Je0gwiMZhBxI1A4cdNrOT6NaqjbTSp5EQD3YUYd9cjaoVSbiuyows/qal1/ur+mgpCXODFncd3wB8SDTcNoW18ZJE1VM52ySr7lmNBcuklJamGpN3n4RLDW7Y787B+pvW+zZsqoWy5xub5bb/N7Hre7bJwe6CSjCVolIeVIJ18XmCiJFvZbUjq119vqS2tmTKdagGVEig9XVJS2nvWHYfKYUe++0NiUC9bTCkVyogVVt3WSBxJuf0rqwSmNSQ52diZLM+jg+lmNRCWB7pq3N1mKVwPu4eixbXAgmyiszVdvZCJjBVKGRp8KSgaUMeQaeTmDperJbKiuzjkq8hsSUKw8LUUoyUD9ZLHRxCfKGmDIaQZ16KEaook5NsIJi5BOKEYWFS0xfyFkJVFpmE6Q4AhGFjS4yRrqVmd3zhuu9iW7WvSaexryGxkCxngh7z6JAYgjvynX5HcInarVaA92OriaxTsIwjRoOtbTHMne8Zfw8SNRFg0gNOgAvDWaJG+1StwPBZRD7uuxQ0EJeHWj0sfMAJUFT5ZpxrrXEI05rMgkrkbomf3kokfTwtAlQCn09giBRcv0Ss1M4Aa2CbFVTZ5sittIFPJzAGx7DFfdEVWhwKnLywDUHppCFoGoClVPvQnC+gqPXJYo5b5TR0fUzFOOUs9tVYFxrfDxHyWccna1FhEldoLB53ZNsqskWyOE+dvWwUJtAFcFAp0g9ysRnU/hhT2pEdl4WfgphgDEGHWwGAVt/wxgExhPhBjl5Lb85A8DNsyzXduIbkYJbTEVIeEvinVwNQuQv03Us6Z4kZOI29sZP3cU6HB2qXtIVG2trBF6A6cwZsuIubzUxxdjEOOldsTo9Ph/r9+2QNiko9DeVYS7N99uoLnPFP7dhrFbWEtc1Yd1d60jghj0AXjcIy3RLEPRz/gxOyV24UeztWV1w7te3opYAoPVkO1NjhochRrwMFsSN5nspKXjye9g23K05t05MGYupNt0L2OTtQppXbtDU7AQU5A/XXXbUBPkUqgzypW6Ye7E8tp0SvqaWTbMawm/6bb65LHwa312whLdjhgf7KCvYTtSLK2365bEHGM9pMq+hbXyKVxvhDhSAcEp3JHiIJyVuWa8SBuDthLGZbAF9GVpCVaA0mA7xyfB+x5l1Ob09E6E8CjnX7AGcZ/ZQE2JyFEHiW5qMlxLYsaDt8rP4G/h+XCKgZOEdyBZTaA/LCN8dFfSaRsqLwUI8FDRfdI2jUhpWFgurqLQMgEpwxp4Ha+8uvB4PM0zfTD0UoNyqlKDMn0AZPahFVjI+nvWGMR3zA9UqOX460Prr663/+IU5v3PVh0wSEssSlR+ICr3frgf8b1nwWIBvSjTpgik5D94WwAW4ZTCF8d6m7G92GTQyL+AtiJsNnqbN1qD6uH7dlxciD7isc+BvR9g0Nl2Pcmfu122ZI6oAiQ8slWwZM2q+vb2EBQA6pUwp1TzH8lDcpit/ow07c4a3YUAA3o5P205nUcI1ZNl+0O1U6huL/Sz98ThuedGGYyBSiFduA6/Ue3/ymyeViuiwsrGJ7DJ5rZ0Ra9i7WQzVNjydDQRwJtLi49qws4bKXGp9nmLlRTU2bqblns1rbEV3Gv7/DQ6v2DQ9YPKEUZ3eT6d4kwWAgeSjZnfgTcxJSqSLb4A1cgVKHBafTbdeWc2IHKUJfR487EIS+jsfU6y/g6zkwIL23eKjPOG2tGMR40mGi2eFu0gM6zYd4PA+yFHLWOsIuuGoaSf/rBQt70lsnb0lr9QSTWRoA8i9IUCH21NSRXHtbRj1JtmG5WuHz+/KUJrLCaUvungYzNEotOOighBPBMokKr3ZZ2iI7DieIwCbBbUCDdtbYpU8W5KHXt2ZTIBP/2BwwwNwclnjdI58lU9NtW2D35MJ7tMwctoJiVLoY57PX3/CEIku9P4RsOMPQQV6Laqwem+Hxognztyvd0+KDXmdBBCuDNz10W0FN3fvFrq8jf0B4D2ajlqjQAaSqgLwv0Pe76v1bwyGkLypO8aJxTa4VgTC6vzvyWvKxHSHNyaSXDFoaoe2odg2SbsYKS2eJzWwPyQrt1lGNXQQUnCZURvYE2e/TEMBzX2QZVRNd11IBShB3RpaJEGuiSUBp0KspjiXZRzNU78M1d3l4tHp671LzEsiXWVLjChBJM8GlJeIFKkxHaiawe+eFGnAggIrVkSTLqAwHcfXyMN5WH8ch9vvSnb1imIFnangh0hR0p9Hp9+Tc8p395fh2RWJrk26UQ6h9VWdfBrPqikcpu1aOgx35J/upE8V3Zo0YpYC5Sz80IJgMQ56qtHbocxpYxT2pxMQwD63qUCaQaaomWdp7+xtrCMha43r4/eDhv0OijWwWOiS23few1qWY0x12SBPm8jCYFVF3avB/TGv6Zdl/qguq0cWU8RJiOAOz0Rx8vOHqKu3fju9HXvfweQZDev3YEta7Q83O3gNL5uEHLpuUv3e5qza6bF5Qc/eOzcfYjZDu6FPiBAgLYpnY0s8D5b+oaUle1iUjNSvk+n5VUu7hzsTvF7vjky31QeYNomtBLWgknRsuQvQhHbe7wb8VnXE7/ZJ/Kd+N+3RIo5Lk+mV43jRiGJIUj0lfEi2Tznmavo72sCwFhanRJzkThuc2J7ca/pHeEE2p6AYrrf8zs0H1HyM+w1dKSCdUql2jQfc3h5+Sh/msbJ2Cdvy0DYbDKRd/mD8MsQUrIeVdasTT8HV44LEr0a8f8YkYfVAXCeebUdyTuPPDSIrhDH2alCJEiaZgBhFPwh9pKoZOEFFWpvUXbTPqxaKfkan1S0tZXkYwZX69d/XAGc4iYWkWpfyZBv5ncr1DWFHiUyjYlWjZz1t2mi0GygpbMM6wc0b8ajzWVsQ4nH2vyYQ6nANP36sZD3iwwoxmJ5tJWEGvTpMt/zQB63cJqk6wyFlPCi8pe3+4DPVeRzWYfLwLaJBcMg7cInavyEkotFPxvt6pfNg759GMtEGggKeDpu7TKeYH/x/ceh2xqtjfuhyjr66fMYdAbdRsx01YY1da+aWtwVQK8XuvbpvgLqf5htDTr/ThNdE11C1JPEtlGfH+zS7JsSjpmRADbSIaZNpuz0NR0WpdYIF3BpFqqS1Qsje3+bhhnAWUBoUQkWpSQHbPRvEgD3mnwnYJkVqMnE/Lh4Aq3/1j23+y0e4SBHeMU80uWtEuWt/+ytukacFHdJCOFmeFlDzJemQi/haG5KVR8rauU6hPKrQ4pHyiwTXQJzGfvKSXa4LRd4IFSE31+/h/vYFnFyTjkPdgyQHbrlTQJfinxeJ88OiUwvam9idBToHMHlRpVDuVU/M+iRJ+YQbFYKfV2lNL9BcIctw6Sd+9Vv9yynoOP+FGy9cWdaUKSW398O3SLG0YtGi6y5VN0kx1/5ye1wrjwWEy6k0jf2LKY+ZetYW4VOlLz1MuDuVCAgFDeedJqfU4v1Im0GO/7TB7ckCSWJSDmSfQqRjzNcbwLMoNxOQv8s3M0zjySHhxnKmwo1RD2MwT5vM/dil291ZafZCXcrtINMFyqfZQZsD6cVPeFfrp5e0hR50nYpIpgcXDz+8QP2qXow7HMYVZNou9FgW8wnweaYOQ6q+vcIv6xCMQnnbxOzju8VGXyavuqYk6IgfMHj6SNBdJviKT6bXA7WQAqnDx0OD2NR6nULj0+G49WtqpnXYPB+xOY3L7Dqg2LYzn1RkkoanG2qXpk9M0Yl1x5sebSIyWobd0+BAcfWTGc4KXQnDto7QbXfQUzW+zlAFBHeBH1Aok0OwjmjPoZSVdLaEn8kvU+8xjgaBQAm2YGKaNKCMGIMNrK4zFfyxr2rKy4RWJriK55q2c4S40JUjows9A30Dokr9YQsQJ5w7RFcEKoU9YPBqyU+3bq8eIEfgAGw5Fa2a5Ez25fsPBmPLgcU/MDy1IZKeW+3pdyeYCiRWrRhUTU2+VZeKzZnXOPRr0iBXQcvEN1DDMg2uDJ49Nfpxq/4n7ghepATABw+ncLcLr2HMX+hDlAZ0BHIcnc6GQh0UHYaheHnoY5+Pywob8FRz29AXCdTYx8anheISMQV5i4bxKMHb1/u4uF3iLYH2EAdtBAyWurILq1KfnS2jhCqghBR8LZexO2gVS7ei0d0DJTRM53npz3PMQzAXARpUgj6F8ABmy/sOxYpYjFwN9QMyuvYf1bBK45p1KrdlmAeml8QM8zYv4+/55mVMIUOsX/T1UqbaTso7SBcqe7C2rCWRI2TtkF58mm0ByrCqx2HvkGVb1AVcnqzb4onbYN4OzQuhZLl5m2De/KvxabyO23D2gh3ck6mUa4ylYkbt8Gnc4Pt33w7vxNrQqdmdDWCMiulesS4cd2vCBxEyTwtU0h804+pw2F5oHbTS1Tm73mFwAe2Cem5oGLbzz+569c078uR7ttjhkXVDGe8/Avqm5UbfTZEs6zDBonExsq1iX4dmZqLhmv0STNm6TG6hMxQolSU3z3KofBmFoykQuyDAuwS3X5Fu0zGpdfQlQ4uP88Jd4ds5DxA+uOzQFFD6aPiozWJNhlUgNulCMJ2W31/FElAJPANYoJgPI0nsoBvZHBTd32EBh7OrKl6vciuYjPVUkkCgey1oHY/DT1zloBuozZi5Rei/LN+VA24Pa7asYhlnwBQLl5C6LlGCXm3DXhhKaDJC5DNkHfiMMh4ql++Qw/jL0yA6F0O2TSomhtxNElINpgIcosC79DA9aJnFfbG7MjGiyoH7ATElwyQ3OUpxqaBdBnEHayL84fyIIVim4OWkhoPf2una684G3YWv76H0g2UCrz4NrhmM+RTbhUR9SRHYf3KZRH2RGyOi4CREKGNpifd5f2HVpjsVHDiygpjYHBmcU/z4DV1IGwakaUoD0xow8My3ls8fLXD94SJfRlww4YJA4La2ZXxc72/VtrSByFTYXACY+DFTqiEUA5JARZ7m4wJUlo7X+Xg1JtPdGzMJFv4xm8P2YOZCakvbck6Wfae5DHRrjGFFvRvINKG50CiTc+rSULPXxIhvZAqt9KFzvzh3zH5NhcLLLkpdRBwST92g6BMQDt7MThU/hLTfnEpZSCjsxqcNXTyIRHTxzahb7aDoqhoiBvCLBoejFtKXBS6R3r0LQKpLFGAb2hGBK2TR4n/VgJIJvmlSj77z9QEm2EgfgmrCo1CjWtwMx6Pqer94gTaLElmb2IPx44Z8qxl/yThZmgXJOFLtn25j8dXQrGhs0nPlhOILpHRsWjE9pxvtLn66PawnT/ToDx/qNhdAwsw5V7sXkG/hLp+KQrxjD900LJF103+7oqFPhSQGoQplRofPx1mTbvSvb1P0YYoS3NGDie/O6x7nIYVrymnh09Ng4CKnKcnoj/ArIhJXk5naIfynz81Nys08Wrtg6ZSbZUfsaOw8lAyNhMbzpuLerBWwbJF0rxFFNd72vsIwHaWeJir3PT4PXUtEXXAkaUNd2uSnXwbXC55h/ETGOwo8U7gwV8XxO+/p/ed4hY9fdDlRqIVhdixHJAHjYm+EZGybLvNA4knfMq0FMY2fXVG0Y5iX0xwTMgehpkP+VjeRrSGQK3pb6JDS7yFIWYX6alB58TFv7+d+/EVPu5fS8fbiGymE5YPSO1n6WBVS8Tey8sd5npcweEzCWCe2KcL5B+p/+5nkah1YAUhXiRzVjUwYJIs0sg7xoKyEb1W+2bJ3CZOLbGytaxKsy7J+D8RxGAOEpfy8KuKXLwbkS8lOhfEChkMOTglXlJd4dsU1BZRPIOx608iCdTIDYboQm4LSOXK35I/Nc6GD6tj2EL7jBHxLeivh+wofvik+yVczwRbPleis6ev2WSVNsPFJESgpg9u7jevqvxZ1Zc7vvkrDGRnMhtKIZLIiJ/hsnXYlQ2Kx+HyJsh3TGX2/EOkqtgDOfNG807On293T4SLqyRKWhwLvRf0FIqjRIV/H7lvjuAh5ZvGdCdEGiubLmhh1qhZkFIqCJ1B2cnHbpwYhfF3AajjTcZ0/wajrcO6nCbpiGjTEqWb6IE3lHpWh4hSUiswBTtQ0YcirMFq4k4iOweu+NpKEDCFcQbb4X9rmWspnIGFZKVQi2rVf5iiDXT4OUYeyuWHHCcW573roOxH4fSP4SYMi2qyhiP+o7s5+j3s8OzuHDDxPw8+xuLrSQUTMA5jkgYhq0rxS2o/T7mVgOAy5DDbTBpJLpyN/VesP177+H0rGA94="
DATA_PATH = Path("auto_mpg.csv")
DATA_PATH.write_bytes(zlib.decompress(base64.b64decode(_DATA)))
plt.rcParams.update({"figure.figsize": (7, 4.5), "axes.spines.top": False,
                     "axes.spines.right": False, "font.size": 11})
print("Ready. Dataset written to", DATA_PATH)

# Part A · Explore and prepare

## The data dictionary
Each row is a vehicle record. Our target is `mpg`.

| Column | Meaning | Use today |
|---|---|---|
| mpg | City fuel efficiency, miles per gallon | Target |
| weight | Vehicle weight, pounds | Numeric predictor |
| horsepower | Engine horsepower | Numeric predictor with missing values |
| displacement | Engine displacement, cubic inches | Numeric predictor |
| cylinders | Cylinder count | Numeric predictor in this lesson |
| acceleration | Acceleration time, seconds | Numeric predictor |
| model_year | Two-digit model year | Numeric predictor |
| origin | Origin category code | Optional categorical extension |
| car_name | Vehicle name | Exclude |

We keep six numeric predictors in the main lab. `origin` is a category code, so we do not treat its numbers as a measured quantity. These choices keep the first pipelines manageable.

## 1. Load the table

**TYPE:** Read `DATA_PATH` with Pandas and save the table as `df`. Display its shape and first five rows.

**Predict first:** what do the two numbers in `.shape` represent?

<details><summary>Small hint — open only after trying</summary>

Use `pd.read_csv(...)`, followed by `.shape` and `.head()`.

</details>

In [ ]:
# TYPE: create df, then inspect its shape and first rows.

In [ ]:
# Run this checkpoint after completing the cell above.
assert "df" in globals(), "Create df in Task 1 first."
assert isinstance(df, pd.DataFrame) and df.shape == (398, 9), "Check the file and loading step."
print("Loaded 398 vehicle records and 9 columns.")

## 2. Inspect before modelling

**TYPE:** Display data types, counts of missing values in every column, and descriptive statistics.

**Write before continuing:** Which column contains missing values? Why would replacing an unknown value with zero change its meaning?

<details><summary>Small hint — open only after trying</summary>

Try `.dtypes`, `.isna().sum()` and `.describe()`.

</details>

In [ ]:
# TYPE: inspect dtypes, missing-value counts, and numeric summaries.

**Your observation:** …

Only inspect structure and basic quality before the split. Use training rows for the relationship plots and modelling decisions that follow.

## 3. Define X and y

**TYPE:** Create a list called `features` containing `weight`, `horsepower`, `displacement`, `cylinders`, `acceleration` and `model_year`, in that order. Use it to make `X`. Save the target column as `y`.

**Explain:** why must `mpg` be excluded from `X`?

<details><summary>Small hint — open only after trying</summary>

`df[features]` selects a table. `df["mpg"]` selects a Series.

</details>

In [ ]:
# TYPE: features = [...], then define X and y.

In [ ]:
# Run this checkpoint after completing the cell above.
assert "X" in globals() and "y" in globals(), "Define X and y first."
assert list(X.columns) == ["weight", "horsepower", "displacement", "cylinders", "acceleration", "model_year"]
assert "mpg" not in X.columns and y.name == "mpg"
assert y.notna().all(), "A supervised training row needs a known target."
print("Six predictors and one numeric target. No target column in X.")

## 4. Reserve the final test set

**TYPE:** Use `train_test_split` to create `X_train`, `X_test`, `y_train`, `y_test`. Reserve **20%** for testing and set `random_state=42`. Print the sizes.

The test set stays closed until Part E. Cross-validation on the training set will guide our choices.

**Predict first:** roughly how many of 398 records will remain for training?

<details><summary>Small hint — open only after trying</summary>

Unpack four outputs. Pass both `X` and `y` to the split function.

</details>

In [ ]:
# TYPE: split X and y, then print training and test sizes.

In [ ]:
# Run this checkpoint after completing the cell above.
assert all(n in globals() for n in ["X_train", "X_test", "y_train", "y_test"]), "Complete the split first."
assert len(X_train) == 318 and len(X_test) == 80
assert set(X_train.index).isdisjoint(X_test.index)
assert X_train.index.equals(y_train.index) and X_test.index.equals(y_test.index)
print("318 training records; 80 untouched test records. Inputs and targets align.")

## 5. Plot one relationship

**TYPE:** Start a fresh chart with `plt.figure()`. Draw a scatter plot of training `weight` on the horizontal axis and `y_train` on the vertical axis. Label both axes with their units.

**Predict first:** as weight increases, do you expect MPG to rise or fall? Is a straight line likely to capture everything?

<details><summary>Small hint — open only after trying</summary>

Use `plt.scatter(...)`, `.xlabel(...)`, `.ylabel(...)`, `.title(...)`, `.show()`.

</details>

In [ ]:
# TYPE: scatter plot, axis labels, title, then show.

**Your interpretation of the plot:** …

# Part B · Build models

A **pipeline** bundles preprocessing with the model. Missing-value statistics must come from the training rows used for each fit. Putting an imputer inside a pipeline also keeps cross-validation folds separate.

`make_pipeline(...)` accepts steps in their execution order and assigns names automatically. You will type the steps. The original `X_train` table remains unchanged. Start each chart in this notebook with `plt.figure()` so it has a fresh plotting area.

## 6. Establish a mean baseline

**TYPE:** Create `baseline` as a pipeline containing `SimpleImputer(strategy="median")` and `DummyRegressor(strategy="mean")`. Fit it on `X_train` and `y_train`.

**Predict first:** will it predict different MPG values for a light car and a heavy car?

<details><summary>Small hint — open only after trying</summary>

Put the imputer first and the regressor second. Call `.fit(X_train, y_train)`.

</details>

In [ ]:
# TYPE: construct baseline with make_pipeline(...), then fit it.

In [ ]:
# Run this checkpoint after completing the cell above.
assert "baseline" in globals(), "Create and fit baseline first."
base_preview = baseline.predict(X_train.iloc[:3])
assert np.allclose(base_preview, y_train.mean())
print("The baseline predicts the training mean for every row:", np.round(base_preview, 2))

## 7. Simple linear regression

**TYPE:**
1. Select `X_train[["weight"]]` into `X_weight`. Double brackets preserve a two-dimensional table.
2. Create `simple` using a median imputer followed by `LinearRegression()`.
3. Fit it on `X_weight` and `y_train`.
4. Predict the first three rows of `X_weight` and print the predictions.

**Explain:** what does `.fit()` do that `.predict()` does not?

<details><summary>Small hint — open only after trying</summary>

A single-bracket Series is one-dimensional. scikit-learn expects a two-dimensional feature table here.

</details>

In [ ]:
# TYPE: define X_weight, build simple, fit, predict three training rows.

In [ ]:
# Run this checkpoint after completing the cell above.
assert "simple" in globals() and "X_weight" in globals(), "Complete Task 7 first."
assert X_weight.shape == (318, 1)
assert simple.named_steps["linearregression"].n_features_in_ == 1
print("One predictor: a fitted straight-line relationship.")

## 8. Put the fitted line on the scatter plot

**TYPE:** Sort `X_weight` by `weight` into `weight_sorted`. Draw the training scatter plot. Add a line using the sorted weights and `simple.predict(weight_sorted)`. Label the axes and show the plot.

**Predict first:** why should we sort the weights before connecting predictions with a line?

<details><summary>Small hint — open only after trying</summary>

Use `.sort_values("weight")` and `plt.plot(...)`. The scatter rows need not be sorted.

</details>

In [ ]:
# TYPE: sort, scatter, plot fitted line, label, show.

## 9. Multiple linear regression

**TYPE:** Build `multiple` using a median imputer, `StandardScaler()` and `LinearRegression()`. Fit it on all six training predictors.

A multiple regression still models a linear relationship in its input features. It can use several predictors at once. Scaling is useful practice for pipelines, though ordinary unpenalised linear regression does not require it.

**Predict first:** does adding predictors guarantee better performance on unseen cars?

<details><summary>Small hint — open only after trying</summary>

The order is impute, scale, then model.

</details>

In [ ]:
# TYPE: make the three-step multiple pipeline and fit it.

In [ ]:
# Run this checkpoint after completing the cell above.
assert "multiple" in globals(), "Create and fit multiple first."
assert multiple.named_steps["linearregression"].n_features_in_ == 6
print("Multiple regression uses six numeric inputs.")

## 10. Polynomial regression

**TYPE:** Build `polynomial` with these steps:
1. Median imputer.
2. `PolynomialFeatures(degree=2, include_bias=False)`.
3. `StandardScaler()`.
4. `LinearRegression()`.

Fit it on `X_weight` and `y_train`.

This creates weight and weight-squared features. It gives a curved relationship with weight while remaining linear in the fitted coefficients.

**Predict first:** will a curve necessarily improve cross-validation performance?

<details><summary>Small hint — open only after trying</summary>

Keep the polynomial expansion on `X_weight`, not the full six-column table.

</details>

In [ ]:
# TYPE: construct polynomial and fit it on the one-predictor table.

In [ ]:
# Run this checkpoint after completing the cell above.
assert "polynomial" in globals(), "Create and fit polynomial first."
assert polynomial.named_steps["polynomialfeatures"].n_output_features_ == 2
print("Two derived inputs: weight and weight squared.")

## 11. Compare line and curve visually

**TYPE:** Draw the training scatter plot. Overlay predictions from `simple` and `polynomial` using `weight_sorted`. Give the fitted lines different colours and labels.

**Explain:** where does the curve differ most from the line? Do these training plots establish which model generalises best?

In [ ]:
# TYPE: scatter, both fitted relationships, labels and legend.

## 12. A decision tree

**TYPE:** Build `tree` using a median imputer and `DecisionTreeRegressor(max_depth=4, random_state=42)`. Fit on all six training predictors.

A tree splits the feature space into regions and predicts a value in each region. It can represent nonlinear patterns without polynomial features. It generally does not need scaling.

**Predict first:** what might happen to training error if the tree could grow without a depth limit?

In [ ]:
# TYPE: build tree and fit it.

## 13. A random forest

**TYPE:** Build `forest` with a median imputer and `RandomForestRegressor(n_estimators=100, min_samples_leaf=2, random_state=42, n_jobs=-1)`. Fit on all six training predictors.

A regression forest averages predictions from many varied trees. Averaging can reduce prediction variability, but a forest is not guaranteed to beat simpler models.

**Predict first:** which is easier to explain to someone: one regression line or 100 trees?

In [ ]:
# TYPE: build forest and fit it.

# Part C · Compare using cross-validation

We split the **318 training rows** into five folds. Each model trains on four folds and predicts the remaining fold. The process repeats until every fold has served as validation data. We reuse the same folds for all candidates.

The imputer and scaler refit within each fold because they belong to the pipeline. The **80 test rows remain unused**.

The helper below handles repetitive scoring code. Read the comments, then run it. You will type the candidate list and the comparison call.

In [ ]:
# Supplied scoring helper; no model answers are included.
cv = KFold(n_splits=5, shuffle=True, random_state=42)
scoring = {"MAE": "neg_mean_absolute_error", "MSE": "neg_mean_squared_error", "R2": "r2"}

def compare_cv(candidates):
    rows = []
    for name, (pipeline, columns) in candidates.items():
        scores = cross_validate(pipeline, X_train[columns], y_train,
                                cv=cv, scoring=scoring, error_score="raise")
        # sklearn negates losses so that a larger score is always better.
        # Undo that sign for readable positive errors.
        fold_mae = -scores["test_MAE"]
        fold_mse = -scores["test_MSE"]
        rows.append({"model": name, "CV MAE": fold_mae.mean(),
                     "MAE fold SD": fold_mae.std(ddof=1),
                     "CV MSE": fold_mse.mean(),
                     "CV RMSE": np.sqrt(fold_mse).mean(),
                     "CV R2": scores["test_R2"].mean()})
    return pd.DataFrame(rows).set_index("model").sort_values("CV MAE")

## 14. Build the comparison

**TYPE:** Create a dictionary called `candidates`. Each entry maps a label to a pair: `(pipeline, column_list)`.

Include these six entries:

| Label | Pipeline variable | Columns |
|---|---|---|
| Mean baseline | baseline | features |
| Simple linear | simple | ["weight"] |
| Multiple linear | multiple | features |
| Polynomial degree 2 | polynomial | ["weight"] |
| Decision tree | tree | features |
| Random forest | forest | features |

Call `compare_cv(candidates)`, save the result as `cv_results`, and display it rounded to three decimals.

**Predict first:** which candidate do you expect to beat the mean baseline?

<details><summary>Small hint — open only after trying</summary>

Dictionary pattern: `"Label": (model_variable, columns)`. Separate entries with commas.

</details>

In [ ]:
# TYPE: candidates = {...}; then create and display cv_results.

In [ ]:
# Run this checkpoint after completing the cell above.
assert "cv_results" in globals() and "candidates" in globals(), "Complete the comparison first."
assert len(cv_results) == 6 and np.isfinite(cv_results.to_numpy()).all()
assert (cv_results[["CV MAE", "CV MSE", "CV RMSE"]] >= 0).all().all()
print("Six models compared using training folds only.")

## 15. Read the table before choosing

Fill these in using your output:

- Lowest cross-validation MAE: …
- Its MAE, in MPG: …
- Baseline MAE, in MPG: …
- Is its improvement large compared with the variation across folds? …
- Does the lowest-MAE model also have the lowest RMSE? …

**Metric reminders**
- **MAE:** average absolute prediction error, in MPG.
- **MSE:** average squared error, in MPG².
- **RMSE:** square root of MSE, in MPG, with more influence from large errors.
- **R²:** compares squared error with predicting the mean of the evaluation observations. It can be negative.

**Which metric should guide a decision?**
- Use **MAE** when every MPG error should count about equally and the result should be easy to explain. It is often a good default for communicating typical error.
- Use **RMSE** when large errors are especially costly or concerning, because squaring gives them extra weight. Report it with MAE when you want both typical and large-error sensitivity.
- Use **MSE** mainly when the squared-error objective is needed for optimisation or mathematical analysis. Its MPG² units make it less intuitive to communicate on its own.
- Use **R²** to describe improvement over a mean-only baseline on the same evaluation data, not as an absolute error measure. A high R² does not guarantee small errors, and R² can be negative.

For this exercise, compare **MAE and RMSE together**, check the baseline, and use R² as supporting context. If MAE and RMSE disagree, inspect the largest residuals and decide whether unusually large errors matter in the application.

The helper reports the **mean of the five fold RMSEs**, not the square root of the mean fold MSE. Fold SD describes variation across these folds; it is not a confidence interval. Different candidate feature sets are intentional: this comparison evaluates whole prediction approaches, not only algorithms.

# Part D · Tune one model and select

**A hyperparameter** is a setting you choose, such as a tree's maximum depth. Model parameters, such as split thresholds, are learned during fitting.

We will try a small tree search using training cross-validation. The search does not use `X_test` or `y_test`.

Trying many settings can make the winning CV score optimistic. This small introductory search illustrates selection; a more rigorous study could use nested cross-validation.

## 16. Search a small parameter grid

**TYPE:**
1. Make `param_grid` with `decisiontreeregressor__max_depth` values `[2, 4, 6, None]` and `decisiontreeregressor__min_samples_leaf` values `[1, 5]`.
2. Create `search = GridSearchCV(...)` using `tree`, the grid, `cv=cv`, `scoring="neg_mean_absolute_error"` and `n_jobs=-1`.
3. Fit on `X_train` and `y_train`.
4. Print `search.best_params_` and **negative** `search.best_score_` to obtain positive MAE.

**Predict first:** how many parameter combinations are there? How many fold fits, before the final refit?

<details><summary>Small hint — open only after trying</summary>

Pipeline parameters use a double underscore: step name, then parameter name. The auto-generated step is `decisiontreeregressor`.

</details>

In [ ]:
# TYPE: param_grid, GridSearchCV, fit, then best settings and MAE.

In [ ]:
# Run this checkpoint after completing the cell above.
assert "search" in globals(), "Complete the grid search first."
assert len(search.cv_results_["params"]) == 8
print("8 combinations × 5 folds = 40 validation fits, plus one refit.")

## 17. Select without opening the test set

**TYPE:** Make `selection_mae` from `cv_results["CV MAE"].copy()`. Add a `"Tuned decision tree"` entry equal to `-search.best_score_`.

Use `.idxmin()` to save the lowest-MAE label as `winner`. Print the sorted scores and the winning label.

Then run the supplied refit cell. Selection uses MAE only. We will not change the winner after seeing test results.

<details><summary>Small hint — open only after trying</summary>

Use `.loc["Tuned decision tree"]` to add an entry to the Series. `.idxmin()` returns the label with the smallest value.

</details>

In [ ]:
# TYPE: selection_mae, tuned-tree entry, winner and printed results.

In [ ]:
# Supplied refit. It fits the chosen approach on all training rows.
assert "winner" in globals(), "Choose winner from validation scores first."
if winner == "Tuned decision tree":
    selected_pipeline = clone(search.best_estimator_)
    selected_columns = features
else:
    chosen_pipeline, selected_columns = candidates[winner]
    selected_pipeline = clone(chosen_pipeline)
selected_pipeline.fit(X_train[selected_columns], y_train)
print("Final training fit complete:", winner)

# Part E · Open the test set once

Freeze your model choice now. The next results describe one final held-out assessment. If you use these results to make further changes, this set has become development data and you need fresh independent data for another final assessment.

## 18. Predict and compute the four metrics

**TYPE:**
1. Predict using `selected_pipeline` and `X_test[selected_columns]`, saving `test_pred`.
2. Create `test_mae`, `test_mse`, `test_rmse`, `test_r2` using the imported metric functions and `np.sqrt`.
3. Print all four with clear labels.
4. Also compute `baseline_test_mae` using the already-fitted `baseline` and the full `X_test`.

**Predict first:** should you expect the test error to be identical to the cross-validation error?

<details><summary>Small hint — open only after trying</summary>

The metric arguments are actual values first, predictions second. For R², reversing them changes the meaning.

</details>

In [ ]:
# TYPE: final predictions, four metrics, and baseline test MAE.

In [ ]:
# Run this checkpoint after completing the cell above.
assert all(n in globals() for n in ["test_pred", "test_mae", "test_mse", "test_rmse", "test_r2", "baseline_test_mae"]), "Compute all requested results first."
assert len(test_pred) == 80 and np.isfinite(test_pred).all()
assert np.isclose(test_rmse**2, test_mse)
assert test_mae >= 0 and test_rmse >= test_mae - 1e-10
print("80 predictions; MAE, MSE and RMSE are consistent.")

## 19. Actual versus predicted

**TYPE:** Make a scatter plot with `y_test` horizontally and `test_pred` vertically. Use `lims = [min(y_test.min(), test_pred.min()), max(y_test.max(), test_pred.max())]` and plot `lims` against `lims` as a dashed reference line. Label the axes in MPG.

**Predict first:** where would every point lie if predictions were perfect? Which side of the line contains overestimates?

In [ ]:
# TYPE: actual-versus-predicted scatter and a diagonal reference line.

## 20. Inspect residuals

**TYPE:** Set `residuals = y_test - test_pred`. Plot predictions horizontally and residuals vertically. Add a horizontal zero line with `plt.axhline(0, ...)`.

**Explain:** a positive residual means the model underestimated or overestimated MPG? Look for curved patterns or errors that grow with predicted MPG. A plot without a visible pattern does not prove all modelling assumptions hold.

In [ ]:
# TYPE: residuals, scatter plot, zero line and labels.

## 20a. Regression assumptions and diagnostic checks

Assumptions matter most when interpreting a linear model or its statistical coefficients. They are less strict for tree and forest predictors, but residual diagnostics are still useful for finding systematic errors.

- **Linearity:** for a linear model, the average relationship between predictors and the target should be reasonably represented by a straight-line combination. Curvature in a relationship plot or residual plot suggests trying a transformation, polynomial term or nonlinear model.
- **Independent observations:** rows should not be repeated measurements of the same vehicle or linked records whose errors influence one another. Random splitting can be misleading when rows are grouped or ordered in time.
- **Constant error spread (homoscedasticity):** residuals should have a roughly similar vertical spread across fitted values. A funnel shape suggests that error size changes with MPG.
- **Residual independence and unusual points:** clusters, runs, or extreme residuals deserve investigation. An outlier may be a valid rare vehicle, a data-entry problem, or evidence that the model misses an important feature.
- **No severe multicollinearity:** strongly related predictors can make individual linear coefficients unstable and difficult to interpret, even when predictions remain useful. Check correlations or use domain knowledge; scaling does not remove multicollinearity.
- **Reasonable missing-data handling and no leakage:** preprocessing must be learned from training data only, and every predictor must be available before prediction time. The pipelines in this notebook protect the imputer during cross-validation.

These checks do not prove that assumptions hold. Use the scatter plots, residual plot, cross-validation results and knowledge of how the data were collected together. For this historical teaching dataset, the goal is to identify important warnings rather than claim that every assumption is perfectly satisfied.

**Discuss:** Which pattern in your residual plot would make you distrust a single linear model? Would that pattern change your choice of metric or model?

## 21. Explain your model to a driver

Write **four or five sentences**, using your own output:

1. Which model did you select, and on what evidence?
2. What does its test MAE mean in MPG? Explain why it is not a maximum error or a guarantee for an individual vehicle.
3. How does it compare with the baseline on the same test records?
4. What does one of your plots reveal?
5. Why would this historical dataset provide limited evidence for modern cars?

**Your explanation:** …

### Read one coefficient
Run the supplied cell below for the simple regression, regardless of which model won. Weight is unscaled in `simple`, so we can report the change per **1,000 pounds**. This is a fitted association with MPG, not proof that changing weight alone would cause that change.

In [ ]:
# Supplied interpretation prompt. This uses the fitted simple model.
slope = simple.named_steps["linearregression"].coef_[0]
print("Fitted MPG change per additional 1,000 pounds:", round(slope * 1000, 2))
print("Explain the sign and magnitude in a sentence. Avoid a causal claim.")

**Your coefficient interpretation:** …

## Exit ticket — close the hint panels

Answer without scrolling back:
1. What is the difference between simple and multiple linear regression?
2. Why is polynomial regression capable of a curved fit?
3. Which dataset rows may teach an imputer its median during cross-validation?
4. Why can a flexible tree have excellent training performance and weaker validation performance?
5. What does the final test set tell us that training performance cannot?

**Your answers:** …

Before submitting, save the notebook with your code, plots and explanations. After filling the required cells, restart the runtime and run all cells to check that the work is reproducible. Re-running a frozen final evaluation is fine; using it to choose changes is the issue.

# Part F · Optional extensions

These are training-data experiments. Keep the final test results frozen.

### Challenge 1: Overfitting you can see
Fit trees with `max_depth` values `1`, `4` and `None`. Use training R² and training-only cross-validation R². Which depth memorises more of the training data? Does that translate into better validation performance?

### Challenge 2: Polynomial degree
Compare degrees `1`, `2` and `5` on weight using training cross-validation. Scale weight before the polynomial expansion for this higher-degree experiment to reduce numerical problems. Explain why the highest degree need not win.

### Challenge 3: Categorical origin
Add `origin` as a category using a `ColumnTransformer`. Apply median imputation to numeric predictors and `OneHotEncoder(handle_unknown="ignore")` to origin. Keep the complete transformer inside the pipeline. Compare using training cross-validation. Do not treat codes 1, 2 and 3 as geographic distances.

### Challenge 4: One change from memory
In a new blank code cell, rebuild a simple regression pipeline and explain every line to your partner without copying an earlier cell.

In [ ]:
# Optional extension: type your experiment here. Keep X_test and y_test out of it.

## Sources and reproducibility

- Quinlan, R. (1993). **Auto MPG**. UCI Machine Learning Repository. DOI: [10.24432/C5859H](https://doi.org/10.24432/C5859H). [Dataset page](https://archive.ics.uci.edu/dataset/9/auto+mpg).
- Data license: [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/).
- Embedded data: the 398-record `auto-mpg.data` file. Preparation only assigns column names, parses `?` as missing and converts the table to CSV. No records are dropped, imputed or scaled in the embedded copy.
- Documentation: [scikit-learn common pitfalls](https://scikit-learn.org/stable/common_pitfalls.html), [cross_validate](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.cross_validate.html), [GridSearchCV](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.GridSearchCV.html).

The fixed seeds make this exercise repeatable within an environment. Results can differ slightly across library versions. The notebook uses standard NumPy, Pandas, Matplotlib and scikit-learn APIs and does not require additional dataset packages.